In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nilanjansaha123/oxford-oncology/Oxford Handbook of Oncology 4th edition (www.myuptodate.com).pdf


In [2]:
print('hello world')

hello world


In [3]:
!pip install -q chromadb pymupdf sentence-transformers langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 77.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 72.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 83.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's 

In [4]:
# =============================================================================
# Oxford Handbook of Oncology -> ChromaDB pipeline (Kaggle, GPU)
#
# Split this into notebook cells at each "# %%" marker (copy-paste per cell,
# or use VSCode/Jupytext "Run Cell" if you're editing this file directly).
# =============================================================================

# %% [Cell 1] Install dependencies
# !pip install -q chromadb pymupdf sentence-transformers langchain-text-splitters

# %% [Cell 2] Imports & config
import os
import re
import glob
import time
import shutil
from collections import Counter

import fitz  # PyMuPDF
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

# --- Locate the PDF robustly (the given path had markdown-link artifacts in it) ---
DATASET_DIR = "/kaggle/input/datasets/nilanjansaha123/oxford-oncology"
_candidates = glob.glob(os.path.join(DATASET_DIR, "*.pdf"))
if not _candidates:
    raise FileNotFoundError(f"No PDF found under {DATASET_DIR} — check the dataset mount path.")
PDF_PATH = _candidates[0]
print(f"Using PDF: {PDF_PATH}")

WORKING_DIR = "/kaggle/working"
CHROMA_DIR = os.path.join(WORKING_DIR, "chroma_db")
COLLECTION_NAME = "oxford_oncology"
EMBED_MODEL_NAME = "abhinand/MedEmbed-large-v0.1"

CHUNK_SIZE_TOKENS = 400       # headroom under the model's 512-token limit
CHUNK_OVERLAP_TOKENS = 60
EMBED_BATCH_SIZE = 128        # lower to 32-64 if you hit a CUDA OOM
DB_INSERT_BATCH_SIZE = 500

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

t_start = time.time()

# %% [Cell 3] Extract text page-by-page
def extract_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in tqdm(range(len(doc)), desc="Extracting pages", unit="page"):
        text = doc[page_num].get_text("text")
        text = re.sub(r"-\n", "", text)          # rejoin hyphenated line breaks
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            pages.append({"page": page_num + 1, "text": text})
    doc.close()
    return pages

pages = extract_pdf_text(PDF_PATH)
print(f"Extracted {len(pages)} non-empty pages")

# %% [Cell 4] Strip repeated running headers/footers
def strip_running_headers_footers(pages, min_page_frac=0.3, max_line_len=60):
    line_counts = Counter()
    for p in pages:
        for line in set(l.strip() for l in p["text"].split("\n") if l.strip()):
            line_counts[line] += 1
    threshold = max(3, int(len(pages) * min_page_frac))
    boilerplate = {ln for ln, cnt in line_counts.items() if cnt >= threshold and len(ln) <= max_line_len}
    for p in pages:
        p["text"] = "\n".join(l for l in p["text"].split("\n") if l.strip() not in boilerplate)
    print(f"Removed {len(boilerplate)} recurring header/footer lines")
    return pages

pages = strip_running_headers_footers(pages)

# %% [Cell 5] Recursive, tokenizer-aware chunking
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=CHUNK_SIZE_TOKENS,
    chunk_overlap=CHUNK_OVERLAP_TOKENS,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for p in tqdm(pages, desc="Chunking pages", unit="page"):
    for i, chunk_text in enumerate(splitter.split_text(p["text"])):
        chunk_text = chunk_text.strip()
        if chunk_text:
            chunks.append({"id": f"p{p['page']}_c{i}", "text": chunk_text, "page": p["page"]})

print(f"Created {len(chunks)} chunks from {len(pages)} pages")

# %% [Cell 6] Load MedEmbed-large
model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print(f"Loaded {EMBED_MODEL_NAME} | dim={model.get_sentence_embedding_dimension()} | device={DEVICE}")

# %% [Cell 7] Embed all chunks (progress bar shows rate + ETA)
texts = [c["text"] for c in chunks]

embeddings = model.encode(
    texts,
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # so cosine similarity == dot product
    device=DEVICE,
)
print(f"Embeddings shape: {embeddings.shape}")

# %% [Cell 8] Build the ChromaDB collection
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

ids = [c["id"] for c in chunks]
metadatas = [
    {"page": c["page"], "source": "Oxford Handbook of Oncology, 4th ed."}
    for c in chunks
]

for start in tqdm(range(0, len(chunks), DB_INSERT_BATCH_SIZE), desc="Writing to ChromaDB", unit="batch"):
    end = start + DB_INSERT_BATCH_SIZE
    collection.add(
        ids=ids[start:end],
        documents=texts[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end],
    )

print(f"Collection '{COLLECTION_NAME}' now holds {collection.count()} vectors")

# %% [Cell 9] Zip the DB for one-click download from Kaggle's Output tab
zip_base = os.path.join(WORKING_DIR, "oxford_oncology_chromadb")
shutil.make_archive(zip_base, "zip", CHROMA_DIR)
zip_size_mb = os.path.getsize(zip_base + ".zip") / 1e6
print(f"Zipped DB ready: {zip_base}.zip  ({zip_size_mb:.1f} MB)")

print(f"\nTotal pipeline time: {time.time() - t_start:.1f}s")

Using PDF: /kaggle/input/datasets/nilanjansaha123/oxford-oncology/Oxford Handbook of Oncology 4th edition (www.myuptodate.com).pdf
Device: cuda (Tesla T4)


Extracting pages:   0%|          | 0/898 [00:00<?, ?page/s]

Extracted 878 non-empty pages
Removed 0 recurring header/footer lines


config.json:   0%|          | 0.00/742 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

/tmp/ipykernel_58/2933315309.py:102: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Loaded {EMBED_MODEL_NAME} | dim={model.get_sentence_embedding_dimension()} | device={DEVICE}")


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embeddings shape: (1441, 1024)


Writing to ChromaDB:   0%|          | 0/3 [00:00<?, ?batch/s]

Collection 'oxford_oncology' now holds 1441 vectors
Zipped DB ready: /kaggle/working/oxford_oncology_chromadb.zip  (13.6 MB)

Total pipeline time: 130.1s


# Retreive

In [5]:
# =============================================================================
# Retrieval example — run this after build_oxford_oncology_chromadb.py has
# created chroma_db/ (in Kaggle), or after you've unzipped
# oxford_oncology_chromadb.zip locally.
# =============================================================================

# !pip install -q chromadb sentence-transformers

import chromadb
from sentence_transformers import SentenceTransformer

# Point this at wherever chroma_db/ lives:
#   - inside the same Kaggle notebook: "/kaggle/working/chroma_db"
#   - after downloading + unzipping locally: "./chroma_db"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "oxford_oncology"
EMBED_MODEL_NAME = "abhinand/MedEmbed-large-v0.1"

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)
model = SentenceTransformer(EMBED_MODEL_NAME)  # add device="cuda" if available


def retrieve(query: str, k: int = 5, page: int | None = None):
    """
    Embed `query` with the same model used to build the DB and return the
    top-k most similar chunks. Pass `page` to restrict to a single page.
    """
    query_vec = model.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_vec,
        n_results=k,
        where={"page": page} if page is not None else None,
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        hits.append({"text": doc, "page": meta["page"], "distance": dist})

    return hits


if __name__ == "__main__":
    # Example: retrieve context for a given patient/cancer scenario
    query = "first-line treatment options for stage III non-small cell lung cancer"
    for hit in retrieve(query, k=5):
        print(f"[page {hit['page']}] distance={hit['distance']:.3f}")
        print(hit["text"])
        print("-" * 80)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[page 264] distance=0.242
240
Chapter 13  Thoracic malignancies
Stage III disease
About 30% of NSCLC patients present with locally advanced disease, but 
with no evidence of metastases. This is a heterogeneous group, with an 
overall poor prognosis, but includes some patients who may have durable 
responses to appropriate radical therapy. In general, responses to chemo­
therapy are more frequent in localized, compared with metastatic, disease, 
but the benefits of chemotherapy are again limited to patients with a good 
performance status.
The same cisplatin or carboplatin chemotherapy doublets are used, with 
the following principles:
•	 response to chemotherapy should improve symptoms, reduce 
loco-regional disease prior to local therapy, and reduce microscopic 
metastatic disease
•	 platinum-based combination chemotherapy, followed by radiotherapy, 
gives better survival rates than radiotherapy alone (3y survival of 
13–23% versus 6%)
•	 a commonly used regimen for patients with PS 0